# HW Y · 03 — Round 3: model soups (weight averaging), still one network

A model soup (Wortsman et al., ICML 2022) averages the *weights* of several fine-tuned copies of one network. All our
ConvNeXt-S runs start from the same ImageNet-22k checkpoint and use LayerNorm, so a per-fold average is well defined.
The result is one network with the same inference cost. The plan, written before any soup was evaluated, uses uniform
soups with no selection:
- **10 full-frame fine-tunes:** 5 runs × raw/EMA weights.
- **6 square fine-tunes:** 3 runs × raw/EMA weights.

Each soup is compared with the Round-2 model by the usual CV rule. `scripts/model_soup.py` writes each soup as an
ordinary run.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HWY_DIR = Path.cwd() if (Path.cwd() / "isic").is_dir() else Path.cwd().parent
sys.path.insert(0, str(HWY_DIR)); sys.path.insert(0, str(HWY_DIR / "research"))
from isic import analysis as A
from isic.data import CLASSES

pd.set_option("display.width", 200)
la = lambda r: f"{r}_tta20_la"
BASE, R1_RECT, ADOPTED = "hwx/convnext_s_kd_res448_tta20_la", "cnx_kd_r1_rect_tta20_la", "cnx_kd_r1_rect_ema_tta20_la"

def run(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=HWY_DIR)

SOUPS = {"soup of 10 full-frame fine-tunes": ("soup_rect10", "ens_rect10_tta20_la",
          ["cnx_kd_r1_rect", "cnx_kd_r2_rect_dp01", "cnx_kd_r2_rect_dp02", "cnx_kd_r2_rect_dp03", "cnx_kd_r2_rect_dp02_s1"]),
         "soup of 6 square fine-tunes": ("soup_square6", "ens_square6_tta20_la", ["cnx_kd_r1_base", "cnx_kd_r1_dp02", "cnx_kd_r1_lld"])}
RUN_SOUP = False   # True on a GPU node: rebuild one soup (a minute) and its TTA-20 evaluation (~18 min)
if RUN_SOUP:
    members = " ".join(f"{m} {m}_ema" for m in SOUPS["soup of 6 square fine-tunes"][2])
    run(f"python scripts/model_soup.py --name soup_square6 --members {members}")
    run("python scripts/evaluate_tta.py soup_square6 --crops 5 && python scripts/posthoc_logit_adjustment.py soup_square6_tta20")

In [2]:
rows = []
for n, (soup, ens, members) in SOUPS.items():
    ing = [la(m) for m in members] + [la(f"{m}_ema") for m in members]
    for label, r in [(f"{n} (1 network)", la(soup)), (f"  its ingredients, mean of {len(ing)}", None), (f"  prediction ensemble of the {len(ing)} ({len(ing)} networks)", ens)]:
        if r is None:
            rows.append({"model": label, "cv": np.mean([A.per_fold(x, "cv_val").bal_acc.mean() for x in ing]),
                         "test": np.mean([A.per_fold(x).bal_acc.mean() for x in ing])})
        else:
            rows.append({"model": label, "cv": A.per_fold(r, "cv_val").bal_acc.mean(), "test": A.per_fold(r).bal_acc.mean()})
rows.insert(0, {"model": "adopted Round-2 model (full frame + EMA)", "cv": A.per_fold(ADOPTED, "cv_val").bal_acc.mean(), "test": A.per_fold(ADOPTED).bal_acc.mean()})
display(pd.DataFrame(rows).set_index("model").round(4))
A.paired_table([la(s) for s, _, _ in SOUPS.values()], ADOPTED, names=[f"{n} vs adopted" for n in SOUPS])

,cv,test
model,,
adopted Round-2 model (full frame + EMA),0.8938,0.8669
soup of 10 full-frame fine-tunes (1 network),0.8799,0.8704
"its ingredients, mean of 10",0.8917,0.8681
prediction ensemble of the 10 (10 networks),0.9010,0.8750
soup of 6 square fine-tunes (1 network),0.8839,0.8690
"its ingredients, mean of 6",0.8877,0.8673
prediction ensemble of the 6 (6 networks),0.8904,0.8825


,cv_bal_acc,cv_mel_sens,cv_auc,test_bal_acc,test_mel_sens,test_auc
soup of 10 full-frame fine-tunes vs adopted,-0.014 ± 0.006 (0/5),+0.011 ± 0.010 (4/5),-0.002 ± 0.001 (0/5),+0.003 ± 0.005 (3/5),+0.040 ± 0.019 (5/5),-0.001 ± 0.000 (0/5)
soup of 6 square fine-tunes vs adopted,-0.010 ± 0.007 (0/5),+0.000 ± 0.017 (2/5),-0.001 ± 0.001 (1/5),+0.002 ± 0.006 (3/5),+0.029 ± 0.018 (5/5),-0.000 ± 0.000 (2/5)


In [3]:
from round3_analysis import bootstrap_ci
lo, hi = bootstrap_ci(ADOPTED)
print(f"adopted model test {A.per_fold(ADOPTED).bal_acc.mean():.4f}, 95% CI {lo:.3f}-{hi:.3f} (baseline {A.per_fold(BASE).bal_acc.mean():.4f})")
pd.DataFrame({"baseline": A.per_class_recall(BASE), "adopted": A.per_class_recall(ADOPTED),
              "soup of 10": A.per_class_recall(la("soup_rect10"))}).round(3).T

adopted model test 0.8669, 95% CI 0.841-0.890 (baseline 0.8639)


,MEL,NV,BCC,AKIEC,BKL,DF,VASC
baseline,0.795,0.873,0.886,0.921,0.823,0.823,0.926
adopted,0.816,0.866,0.873,0.898,0.824,0.855,0.937
soup of 10,0.856,0.802,0.888,0.870,0.813,0.905,0.960


## Takeaways
* **Neither soup passes the rule** (CV −0.014 and −0.010 against the Round-2 model, 0/5 folds). On test, each soup is
  at or slightly above its ingredients' mean (0.870 vs 0.868; 0.869 vs 0.867).
* The CV deficit (about 0.012 below the ingredients) matches the best-epoch selection bonus found in Rounds 1–2. Each
  ingredient is its fold's best epoch; the soup is not selected at all. This is the same bias that hid EMA.
* Prediction ensembles of the same networks reach test 0.875 / 0.883. The gap to them, and to the 0.892 teacher
  ensemble, comes from combining different networks, not from training one network better.
* **Adopted model after 3 rounds:** full frame + EMA, CV 0.894, test 0.867 (95% CI 0.841–0.890). That is +0.003 over the
  baseline, within seed noise.